# v3 Foreign robustness and Dealer divergence

Preserve the frozen v2 grid, run the four bounded v3 modules, then upload the immutable archive to the canonical Google Drive folder. No new Z-score analysis is added.

In [ ]:
REPO_URL = "https://github.com/hh4832/taiwan-futures-oi-research.git"
REPO_DIR = "/content/taiwan-futures-oi-research"
BRANCH = "research/futures-finite-grid-robustness"


In [ ]:
import os
import shutil
import subprocess

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
result = subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR],
    text=True,
    capture_output=True,
)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()
os.chdir(REPO_DIR)
subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Branch:", BRANCH)
print("Commit:", commit)


In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pytest -q


In [ ]:
import os
from google.colab import userdata

finlab_token = userdata.get("FINLAB_API_TOKEN")
assert finlab_token, "FINLAB_API_TOKEN is missing from Colab Secrets"
os.environ["FINLAB_API_TOKEN"] = finlab_token
print("FinLab token loaded from Colab Secrets.")


In [ ]:
import pandas as pd
from src.config import CONFIG
from src.pipeline import run_research

daily, results, archive = run_research(output_root="outputs", inspect_schema=True)
primary = results.loc[results["outcome_role"].eq("primary")]
counts = (
    primary.groupby("institution", sort=False).size()
    .reindex(CONFIG.comparison_institutions)
    .rename("primary_rows")
)
display(counts.to_frame())
assert counts.notna().all(), f"Missing institution results: {counts[counts.isna()].index.tolist()}"
assert counts.gt(0).all(), f"Empty institution results: {counts[counts.le(0)].index.tolist()}"
display(primary.groupby(["institution", "side", "group"], observed=True)[["n", "mean_return", "hac_coef", "q_value_family", "q_value_global"]].mean().head(40))
display(pd.read_csv(archive / "data_quality_report.csv"))
display(pd.read_csv(archive / "global_fdr_summary.csv").query("outcome == 'o1_c1'").head(40))
print("Local archive:", archive)
print("Archive name:", archive.name)
display(pd.read_csv(archive / "prior_return_summary.csv"))
display(pd.read_csv(archive / "incremental_decay_summary.csv"))
display(pd.read_csv(archive / "horizon_nonoverlap_summary.csv").head(30))
display(pd.read_csv(archive / "foreign_dealer_divergence_fdr.csv"))
display(pd.read_csv(archive / "foreign_dealer_divergence_monotonicity.csv"))


In [ ]:
from src.config import DRIVE_ROOT_FOLDER_ID
from src.drive_upload import upload_archive_to_drive

upload_result = upload_archive_to_drive(archive, DRIVE_ROOT_FOLDER_ID)
display(upload_result)
assert upload_result["status"] == "success"
assert upload_result["parent_folder_id"] == DRIVE_ROOT_FOLDER_ID
